In [1]:
import logging
import random
import time
from os import getenv
import subprocess

import fs_structs
from dotenv import load_dotenv

from distributed_processing.async_result import gather
from distributed_processing.utils import fsclient

In [ ]:
load_dotenv()
NS_PATH = getenv("NS_PATH")
LAUNCH_SERVER = True

In [3]:
# logging.getLogger("distributed_processing").setLevel(logging.DEBUG)
# logging.getLogger("distributed_processing.filesystem_connector").setLevel(logging.DEBUG)

In [ ]:
if LAUNCH_SERVER:
    PYTHON_INTERPRETER = getenv("PYTHON_INTERPRETER")
    MULTI_WORKER_SCRIPT = getenv("MULTI_WORKER_SCRIPT")
    server_process = subprocess.Popen(
        [PYTHON_INTERPRETER, MULTI_WORKER_SCRIPT],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        shell=False
    )
    time.sleep(10)

In [6]:
# NBVAL_CHECK_OUTPUT
"""
fs_connector = FileSystemConnector(NS_PATH)
fs_connector.with_watchdog=True
fs_connector.pop_watchdog_timeout = 10

client = Client(DummySerializer(), fs_connector, check_registry="cache")
"""

client = fsclient(NS_PATH)

Client with id: fs_client_1
Results queue: fs_client_1_responses


In [7]:
client.update_registry_cache()

In [8]:
ns = fs_structs.structs.FSNamespace(NS_PATH)

In [9]:
# NBVAL_CHECK_OUTPUT
ns.registry.keys()

['method_queues_create_worker',
 'method_queues_kill_process',
 'method_queues_kill_processes',
 'method_queues_list_processes',
 'nclients',
 'workers_queue_requests_node_1']

In [10]:
z1 = client.rpc_sync("create_worker", ["worker1"])
z2 = client.rpc_sync("create_worker", ["worker1"])
z3 = client.rpc_sync("create_worker", ["worker1"])

(z1, z2, z3)

(37100, 27512, 30884)

In [11]:
ps = client.rpc_sync("list_processes", [])
ps

[(37100, 'worker1', 'fs_server_1'),
 (27512, 'worker1', 'fs_server_2'),
 (30884, 'worker1', 'fs_server_3')]

In [12]:
# NBVAL_CHECK_OUTPUT

[(w, q) for _, w, q in ps]

[('worker1', 'fs_server_1'),
 ('worker1', 'fs_server_2'),
 ('worker1', 'fs_server_3')]

In [13]:
# NBVAL_CHECK_OUTPUT

client.rpc_sync("kill_process", [z2])

True

In [14]:
[(w, q) for _, w, q in client.rpc_sync("list_processes", [])]

[('worker1', 'fs_server_1'), ('worker1', 'fs_server_3')]

In [15]:
client.update_registry_cache()

In [16]:
y = client.rpc_async("add", [1, 0])

In [17]:
# NBVAL_CHECK_OUTPUT

y.get()

1

In [18]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "always"

try:
    y = client.rpc_async("fake", [1, 0])
except Exception as e:
    print(e)

'method_queues_fake'


In [19]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "never"  # use queue client.default_queue, by default "default"
client.set_default_queue("cola_1")

y = client.rpc_async("fake", [1, 0])

try:
    y.get()
except Exception as e:
    print(e)


Error -32601 : The method does not exist/is not available.

 NoneType: None



In [20]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "never"
client.set_default_queue("cola_1")

y = client.rpc_async("fake", [1, 0])

y.safe_get(default="Esto es un error")

'Esto es un error'

In [21]:
client.check_registry = "cache"

In [22]:
x = client.rpc_async("div", [1, 0])

In [23]:
try:
    x.get()
except Exception as e:
    print(e)

Error -3260 : Undefined error

 Traceback (most recent call last):
  File "\\ntcimdavinfo2\fondos\arquitectura_gestora\desarrollo\py_distributed_processing\distributed_processing\worker.py", line 362, in process_single_request
    result = func[request["method"]](*args, **kwargs)
  File "G:\arquitectura_gestora\desarrollo\py_distributed_processing\tests\fs_worker_multi.py", line 28, in div
    return x / y
           ~~^~~
ZeroDivisionError: division by zero



In [24]:
# NBVAL_CHECK_OUTPUT

try:
    x.get()
except Exception as e:
    print("Error")

Error


In [25]:
# x = client.rpc_sync("div", [1, 0])

In [26]:
# NBVAL_CHECK_OUTPUT


client.rpc_sync("add", [1, 1])

2

In [27]:
def f(x, y):
    return x + y


y = client.rpc_async_fn(f, [1, 2.0])

In [28]:
y.get()

3.0

In [29]:
# NBVAL_CHECK_OUTPUT

client.rpc_sync_fn(f, [3.0, 2.0])

5.0

In [30]:
tp = []
N = 30
for i in range(N):
    fn = random.choice(("add", "mul", "div", "lista", "tupla", "dic"))
    t = (fn, [random.random(), random.random()], {})
    tp.append(t)

t = ("sleep", [10], {})
tp.append(t)
tp


[('mul', [0.20508243958574257, 0.1829638120050936], {}),
 ('tupla', [0.5613680394741591, 0.3914366368922122], {}),
 ('div', [0.03710720236117515, 0.05135569656019279], {}),
 ('dic', [0.5238582131040291, 0.9495648072385102], {}),
 ('div', [0.5999641861187863, 0.9005716606194594], {}),
 ('mul', [0.1660821271243489, 0.10483130361434356], {}),
 ('lista', [0.6396763182733894, 0.7064236257937668], {}),
 ('lista', [0.17254001987271483, 0.8482720406038585], {}),
 ('dic', [0.04344174333961104, 0.4341703972558578], {}),
 ('mul', [0.12852553219427487, 0.1432283615890183], {}),
 ('div', [0.4519429272286314, 0.9045160095746618], {}),
 ('tupla', [0.16038495325221325, 0.04744088716773509], {}),
 ('dic', [0.8383338315214117, 0.746252225190962], {}),
 ('tupla', [0.9901906151087763, 0.41813299935969905], {}),
 ('dic', [0.08588539287393582, 0.6936257643186399], {}),
 ('div', [0.7726909781981313, 0.7324413468413015], {}),
 ('dic', [0.7375278719521015, 0.08690869226170672], {}),
 ('mul', [0.957875647099595

In [31]:
tp = [
    ("div", [0.5273328219558507, 0.13835718442890943], {}),
    ("div", [0.7224042937278776, 0.6744424742714074], {}),
    ("mul", [0.16464192867054384, 0.2961055537758295], {}),
    ("lista", [0.24324779336838243, 0.4486736957376778], {}),
    ("dic", [0.222443603731179, 0.049201002693669005], {}),
    ("dic", [0.5892562777042863, 0.8190093828367946], {}),
    ("div", [0.9698052964451762, 0.2495544466990297], {}),
    ("add", [0.18281717945238662, 0.28456253134154685], {}),
    ("div", [0.8231058337900704, 0.4550312056693214], {}),
    ("mul", [0.6955981505190975, 0.2960636895338262], {}),
    ("add", [0.5793774647414703, 0.9353212122782703], {}),
    ("lista", [0.3893530442489298, 0.74231052966268], {}),
    ("div", [0.6419882052325951, 0.7661892480993476], {}),
    ("div", [0.049994104986677, 0.4562378471461709], {}),
    ("dic", [0.4734342157728231, 0.053714834674179035], {}),
    ("mul", [0.8092977961625194, 0.9195146847049076], {}),
    ("mul", [0.913778636227633, 0.6145608354175943], {}),
    ("lista", [0.7499808955353686, 0.5337360859450593], {}),
    ("dic", [0.6756209555432302, 0.9505296005797351], {}),
    ("dic", [0.5209295316393681, 0.9420323858687962], {}),
    ("div", [0.15809810362813237, 0.62619392590883], {}),
    ("dic", [0.29474022335742067, 0.893515494048087], {}),
    ("mul", [0.5349022233942071, 0.05757455715844495], {}),
    ("lista", [0.042102069906052586, 0.3469740971990074], {}),
    ("mul", [0.871021663889528, 0.007201521377221853], {}),
    ("lista", [0.6049724123450363, 0.26801461728942155], {}),
    ("dic", [0.8898534023208522, 0.5019296231298458], {}),
    ("lista", [0.9266421130454301, 0.9972178178045238], {}),
    ("mul", [0.48602513421859184, 0.199817263488683], {}),
    ("dic", [0.7163791721275343, 0.6950721561324937], {}),
    ("sleep", [10], {}),
]

In [32]:
fs = []
for t in tp:
    fs.append(client.rpc_async(t[0], t[1], t[2]))

In [33]:
# NBVAL_CHECK_OUTPUT

resultados = [f.get() for f in fs]
resultados

[3.8113873459661534,
 1.0711132843587332,
 0.048751389463712,
 [0.24324779336838243, 0.4486736957376778],
 {'a': 0.222443603731179, 'b': [0.222443603731179, 0.049201002693669005]},
 {'a': 0.5892562777042863, 'b': [0.5892562777042863, 0.8190093828367946]},
 3.886147128505352,
 0.4673797107939335,
 1.8088997491487095,
 0.2059413548755898,
 1.5146986770197406,
 [0.3893530442489298, 0.74231052966268],
 0.8378976954129118,
 0.10957903930898512,
 {'a': 0.4734342157728231, 'b': [0.4734342157728231, 0.053714834674179035]},
 0.7441612078707556,
 0.5615725620668042,
 [0.7499808955353686, 0.5337360859450593],
 {'a': 0.6756209555432302, 'b': [0.6756209555432302, 0.9505296005797351]},
 {'a': 0.5209295316393681, 'b': [0.5209295316393681, 0.9420323858687962]},
 0.2524746681288481,
 {'a': 0.29474022335742067, 'b': [0.29474022335742067, 0.893515494048087]},
 0.03079675863498907,
 [0.042102069906052586, 0.3469740971990074],
 0.006272681132523783,
 [0.6049724123450363, 0.26801461728942155],
 {'a': 0.8898

In [34]:
# NBVAL_CHECK_OUTPUT

fs = client.rpc_multi_async(tp)

In [35]:
time.sleep(3)

In [36]:
# NBVAL_CHECK_OUTPUT

# AsynResult.status updates the information every time it is called.
# 'PENDING' state should be assumed as transitory.
# If there are responses available in the queue or in the cache
# status or pending() updates the AsyncResult object.

[f.status for f in fs]

['OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'PENDING']

In [37]:
r = client.wait_responses(timeout=2)
r

['fs_client_1:75']

In [38]:
# NBVAL_CHECK_OUTPUT

len(r)

1

In [39]:
time.sleep(7)

In [40]:
# NBVAL_CHECK_OUTPUT
# AsynResult.status updates information

[f.status for f in fs]

['OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK']

In [41]:
# NBVAL_CHECK_OUTPUT

client.wait_responses()

[]

In [42]:
# NBVAL_CHECK_OUTPUT

try:
    client.wait_responses(["kk", "qq"])
except ValueError as e:
    print(e)

wait_responses: ['kk', 'qq'] neither in responses nor in pending.


In [43]:
client._update_cache_with_all_available_responses()

In [44]:
# NBVAL_CHECK_OUTPUT

client.pending

{}

In [45]:
# NBVAL_CHECK_OUTPUT

[f.get() for f in fs]

[3.8113873459661534,
 1.0711132843587332,
 0.048751389463712,
 [0.24324779336838243, 0.4486736957376778],
 {'a': 0.222443603731179, 'b': [0.222443603731179, 0.049201002693669005]},
 {'a': 0.5892562777042863, 'b': [0.5892562777042863, 0.8190093828367946]},
 3.886147128505352,
 0.4673797107939335,
 1.8088997491487095,
 0.2059413548755898,
 1.5146986770197406,
 [0.3893530442489298, 0.74231052966268],
 0.8378976954129118,
 0.10957903930898512,
 {'a': 0.4734342157728231, 'b': [0.4734342157728231, 0.053714834674179035]},
 0.7441612078707556,
 0.5615725620668042,
 [0.7499808955353686, 0.5337360859450593],
 {'a': 0.6756209555432302, 'b': [0.6756209555432302, 0.9505296005797351]},
 {'a': 0.5209295316393681, 'b': [0.5209295316393681, 0.9420323858687962]},
 0.2524746681288481,
 {'a': 0.29474022335742067, 'b': [0.29474022335742067, 0.893515494048087]},
 0.03079675863498907,
 [0.042102069906052586, 0.3469740971990074],
 0.006272681132523783,
 [0.6049724123450363, 0.26801461728942155],
 {'a': 0.8898

In [46]:
fs = client.rpc_batch_async(tp)

In [47]:
# NBVAL_CHECK_OUTPUT

[f.get() for f in fs]

[3.8113873459661534,
 1.0711132843587332,
 0.048751389463712,
 [0.24324779336838243, 0.4486736957376778],
 {'a': 0.222443603731179, 'b': [0.222443603731179, 0.049201002693669005]},
 {'a': 0.5892562777042863, 'b': [0.5892562777042863, 0.8190093828367946]},
 3.886147128505352,
 0.4673797107939335,
 1.8088997491487095,
 0.2059413548755898,
 1.5146986770197406,
 [0.3893530442489298, 0.74231052966268],
 0.8378976954129118,
 0.10957903930898512,
 {'a': 0.4734342157728231, 'b': [0.4734342157728231, 0.053714834674179035]},
 0.7441612078707556,
 0.5615725620668042,
 [0.7499808955353686, 0.5337360859450593],
 {'a': 0.6756209555432302, 'b': [0.6756209555432302, 0.9505296005797351]},
 {'a': 0.5209295316393681, 'b': [0.5209295316393681, 0.9420323858687962]},
 0.2524746681288481,
 {'a': 0.29474022335742067, 'b': [0.29474022335742067, 0.893515494048087]},
 0.03079675863498907,
 [0.042102069906052586, 0.3469740971990074],
 0.006272681132523783,
 [0.6049724123450363, 0.26801461728942155],
 {'a': 0.8898

In [48]:
# NBVAL_CHECK_OUTPUT

client.rpc_batch_sync(tp)

[3.8113873459661534,
 1.0711132843587332,
 0.048751389463712,
 [0.24324779336838243, 0.4486736957376778],
 {'a': 0.222443603731179, 'b': [0.222443603731179, 0.049201002693669005]},
 {'a': 0.5892562777042863, 'b': [0.5892562777042863, 0.8190093828367946]},
 3.886147128505352,
 0.4673797107939335,
 1.8088997491487095,
 0.2059413548755898,
 1.5146986770197406,
 [0.3893530442489298, 0.74231052966268],
 0.8378976954129118,
 0.10957903930898512,
 {'a': 0.4734342157728231, 'b': [0.4734342157728231, 0.053714834674179035]},
 0.7441612078707556,
 0.5615725620668042,
 [0.7499808955353686, 0.5337360859450593],
 {'a': 0.6756209555432302, 'b': [0.6756209555432302, 0.9505296005797351]},
 {'a': 0.5209295316393681, 'b': [0.5209295316393681, 0.9420323858687962]},
 0.2524746681288481,
 {'a': 0.29474022335742067, 'b': [0.29474022335742067, 0.893515494048087]},
 0.03079675863498907,
 [0.042102069906052586, 0.3469740971990074],
 0.006272681132523783,
 [0.6049724123450363, 0.26801461728942155],
 {'a': 0.8898

In [49]:
tp = []
N = 30
for i in range(N):
    fn = random.choice(("add", "mul", "div", "fake"))
    t = (fn, [random.random(), random.random()], {})
    tp.append(t)

tp

[('mul', [0.5282267486385759, 0.06661326115657729], {}),
 ('mul', [0.893690949679695, 0.05779000657872624], {}),
 ('mul', [0.660400700373257, 0.9408172665407235], {}),
 ('mul', [0.9166572420030937, 0.15100029450293528], {}),
 ('mul', [0.6963371367727327, 0.3570351390319041], {}),
 ('add', [0.5790427625075604, 0.719306563620625], {}),
 ('div', [0.3846380786664325, 0.9246221860770829], {}),
 ('add', [0.36727145970698893, 0.14540530269111795], {}),
 ('fake', [0.1926946209042666, 0.49721483498610153], {}),
 ('mul', [0.7195819300929887, 0.013183122296715921], {}),
 ('add', [0.5576408249453472, 0.532446772324449], {}),
 ('add', [0.37574114663589364, 0.11388599944584787], {}),
 ('fake', [0.7582827291317705, 0.7765114930907783], {}),
 ('fake', [0.1168743480063521, 0.032074430630108064], {}),
 ('add', [0.09784930331244746, 0.3716488866396047], {}),
 ('mul', [0.5074804234140887, 0.802111647566446], {}),
 ('fake', [0.3137730970080054, 0.9606608190996937], {}),
 ('div', [0.47357537189398924, 0.195

In [50]:
tp = [
    ("mul", [0.7355407026565478, 0.023519777553893007], {}),
    ("fake", [0.2520522260048764, 0.28054227055242864], {}),
    ("mul", [0.8838106264839363, 0.19639888038506714], {}),
    ("mul", [0.8857412900943293, 0.1253016179972587], {}),
    ("add", [0.0005226378683395039, 0.6568308199617323], {}),
    ("add", [0.15476536347494607, 0.4492869825171584], {}),
    ("fake", [0.7631067253940333, 0.019049359538004906], {}),
    ("fake", [0.4310102131915736, 0.675491507770126], {}),
    ("fake", [0.7140511506543913, 0.7837833237953286], {}),
    ("add", [0.0909525538014786, 0.44959184616881276], {}),
    ("add", [0.6627327150574825, 0.7401973301261011], {}),
    ("div", [0.21232540669537237, 0.7667374101737603], {}),
    ("add", [0.887254961441083, 0.21290364755712576], {}),
    ("fake", [0.7372649371990656, 0.3796617846297834], {}),
    ("add", [0.30864649241428155, 0.8777033159855755], {}),
    ("div", [0.4997997676145608, 0.45884184399530026], {}),
    ("fake", [0.0011733893324340494, 0.6016126834507816], {}),
    ("mul", [0.9307150630961242, 0.2943025202085412], {}),
    ("div", [0.6545197868355805, 0.11276464241684814], {}),
    ("mul", [0.6573680105483782, 0.13653818666573825], {}),
    ("add", [0.7959397704608381, 0.41576468218269147], {}),
    ("mul", [0.16347738503061415, 0.04898545483226813], {}),
    ("fake", [0.7815677830141265, 0.013945854620984632], {}),
    ("fake", [0.8020187012792446, 0.25875661742111045], {}),
    ("fake", [0.9043915109893543, 0.6876522434184933], {}),
    ("add", [0.929922781635924, 0.9540258004221696], {}),
    ("fake", [0.961238888823737, 0.4030318469598062], {}),
    ("div", [0.7466755564012741, 0.799944178434153], {}),
    ("mul", [0.15308290555092918, 0.45297088397324015], {}),
    ("mul", [0.3505532310800745, 0.5551384076337974], {}),
]

In [51]:
client.check_registry = "never"
client.set_default_queue("cola_1")

fs = client.rpc_batch_async(tp)

In [52]:
# NBVAL_CHECK_OUTPUT

[f.safe_get() for f in fs]

[0.017299753708316164,
 None,
 0.17357941751386985,
 0.11098481677579876,
 0.6573534578300718,
 0.6040523459921044,
 None,
 None,
 None,
 0.5405443999702914,
 1.4029300451835836,
 0.27692063003323986,
 1.1001586089982087,
 None,
 1.1863498083998572,
 1.0892637063407844,
 None,
 0.2739117886652408,
 5.804299759281539,
 0.08975583613233945,
 1.2117044526435294,
 0.008008014060514455,
 None,
 None,
 None,
 1.8839485820580935,
 None,
 0.9334095759817276,
 0.06934209904859642,
 0.1946055624926752]

In [53]:
# NBVAL_CHECK_OUTPUT

client.responses

{}

In [54]:
client.check_registry = "cache"

In [55]:
# NBVAL_CHECK_OUTPUT

try:
    x = client.rpc_batch_sync(tp, timeout=5)
except Exception as e:
    print(e)

Method fake does not exist/is not available.


In [56]:
client.check_registry = "never"
client.set_default_queue("cola_1")

x = client.rpc_async("kk", [1, 0])

In [57]:
# NBVAL_CHECK_OUTPUT

try:
    x.get()
except Exception as e:
    print(e)

Error -32601 : The method does not exist/is not available.

 NoneType: None



In [58]:
y = client.rpc_async("add", [1, 0])

In [59]:
# NBVAL_CHECK_OUTPUT

y.get(5)

1

In [60]:
# NBVAL_CHECK_OUTPUT


def f(x, y):
    return x + y


# "never" use queue "default" don't work for rpc_async_fn or rpc_sync_fn
client.check_registry = "never"
y = client.rpc_async_fn(f, [1, 2.0])
try:
    print(y.get())
except Exception as e:
    print(e)

Error -32601 : The method does not exist/is not available.

 NoneType: None



In [61]:
client.check_registry = "cache"
y = client.rpc_async_fn(f, [1, 2.0])

In [62]:
# NBVAL_CHECK_OUTPUT

y.get()

3.0

In [63]:
# NBVAL_CHECK_OUTPUT

[k for k in client.responses.keys()]

[]

In [64]:
client.clean_used()

In [65]:
# NBVAL_CHECK_OUTPUT

[k for k in client.responses.keys()]

[]

In [66]:
#client.rpc_sync_fn(print, ["hola"])

In [67]:
# NBVAL_CHECK_OUTPUT

tp = [
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
]
tp

[('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {})]

In [68]:
# NBVAL_CHECK_OUTPUT

client.default_requests_queue

'requests_cola_1'

In [69]:
client.check_registry = "never"
fs = client.rpc_multi_async(tp, retry=True)

In [70]:
# NBVAL_CHECK_OUTPUT

gather(fs, 30, 5)

In [71]:
s = [f.status for f in fs] 

In [72]:
# NBVAL_CHECK_OUTPUT

len(s), "OK" in s

(10, True)

In [73]:
[(time.time() if f.finished_time is None else f.finished_time) - f.creation_time for f in fs]

[15.127023935317993,
 15.115586757659912,
 30.129367113113403,
 30.117406368255615,
 30.983983993530273,
 30.958498001098633,
 30.934165477752686,
 30.912595510482788,
 30.895402669906616,
 30.87813711166382]

In [74]:
[f.finished_time for f in fs]

[1774025076.2818525,
 1774025076.2922902,
 1774025091.3246825,
 1774025091.3354502,
 None,
 None,
 None,
 None,
 None,
 None]

In [75]:
client.pending

{'fs_client_1:176': 1774025061.240999,
 'fs_client_1:177': 1774025061.266486,
 'fs_client_1:178': 1774025061.2908192,
 'fs_client_1:179': 1774025061.3123915,
 'fs_client_1:180': 1774025061.3295832,
 'fs_client_1:181': 1774025061.3468502}

In [76]:
fs[3].status

'OK'

In [77]:
fs[4].retry()

True

In [78]:
# [f.retry() for f in fs if not f.done()]
# no hace falta chequear si está pendiente.
[f.retry() for f in fs]

[False, False, False, False, True, True, True, True, True, True]

In [79]:
client.check_registry = "always"

In [80]:
# NBVAL_CHECK_OUTPUT

sorted(client.connector.all_queues_for_method("info"))

['requests_fs_server_1', 'requests_fs_server_3']

In [81]:
client.update_registry_cache()

In [82]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "Never"
client.all_queues_for_method("hola")

['requests_cola_1']

In [83]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "always"
a = client.rpc_async("info")

In [84]:
#client.rpc_sync("div", [2, 1], timeout=10)

In [85]:
x = a.get()
x

('fs_server_1',
 {'requests_fs_server_1': {'info'},
  'requests_cola_1': {'add', 'dic', 'div', 'lista', 'mul', 'sleep', 'tupla'},
  'requests_cola_2': {'hola'},
  'requests_py_eval': {'eval_py_function'}})

In [86]:
# NBVAL_CHECK_OUTPUT

len(x), len(x[1])

(2, 4)

In [87]:
# NBVAL_CHECK_OUTPUT

x[1]['requests_cola_1']

{'add', 'dic', 'div', 'lista', 'mul', 'sleep', 'tupla'}

In [88]:
# NBVAL_CHECK_OUTPUT

x[1]['requests_cola_2']

{'hola'}

In [89]:
# NBVAL_CHECK_OUTPUT

x[1]['requests_py_eval']

{'eval_py_function'}

In [90]:
# NBVAL_CHECK_OUTPUT

client.default_requests_queue

'requests_cola_1'

In [91]:
client.set_default_queue("cola_1")

In [92]:
# NBVAL_CHECK_OUTPUT

client.default_requests_queue

'requests_cola_1'

In [ ]:
if LAUNCH_SERVER:
    server_process.terminate()
    server_process.wait()